# Lab 8. Mạng Nơ-ron Tích Chập (CNN)

Notebook bám sát các cell thực hành trong file `Lab8. CNN_Park1.pdf`. Các câu hỏi trong đề được giữ lại để trả lời sau khi chạy notebook và có output.

## 1. GIAI ĐOẠN 1 - WOW phase: Cho CNN chạy thật

### 1.1. Bước 1 - Chuẩn bị môi trường

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

import random
from pathlib import Path
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang dùng: {device}")

DATA_DIR = Path('/kaggle/working/data') if Path('/kaggle/working').exists() else Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

**Câu hỏi 1.1**

(a) Output có in ra `cuda` không? Nếu in `cpu`, vào menu Runtime của Colab/Kaggle và chuyển sang GPU. Tại sao GPU lại nhanh hơn CPU rất nhiều khi train CNN?

(b) `torch.manual_seed(42)` dùng để làm gì?

### 1.2. Bước 2 - Tải dữ liệu MNIST và xem thử

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=str(DATA_DIR), train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print(f"Batch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")
print("Classes:", train_set.classes)
print("Train size:", len(train_set))
print("Test size:", len(test_set))

fig, axs = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axs.flat):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f"label = {labels[i].item()}")
    ax.axis('off')
plt.tight_layout()
plt.show()

**Câu hỏi 2.1**

(a) Shape của images là `torch.Size([64, 1, 28, 28])`. Đoán xem 4 con số này nghĩa là gì?

(b) Tại sao là 1 kênh màu mà không phải 3?

(c) `train_set.classes`, `len(train_set)`, `len(test_set)` cho biết gì? Có bao nhiêu ảnh train, bao nhiêu ảnh test?

### 1.3. Bước 3 - Tạo một CNN nhỏ

In [ ]:
class TinyCNN(nn.Module):
    """CNN tối giản 2 lớp tích chập + 1 lớp fully-connected."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = TinyCNN().to(device)
print(model)
print(f"Tổng tham số: {sum(p.numel() for p in model.parameters()):,}")

### 1.4. Bước 4 - TRAIN

In [ ]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

print(f"Accuracy trước khi train: {evaluate_accuracy(model, test_loader):.2f}%")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_acc_history, test_acc_history = [], []

for epoch in range(5):
    model.train()
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    test_acc = 100 * correct / total
    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)
    print(f"Epoch {epoch+1}: train_acc = {train_acc:.2f}% | test_acc = {test_acc:.2f}%")

**Câu hỏi 4.1**

(a) Accuracy ban đầu trước train là khoảng bao nhiêu phần trăm? Vì sao?

(b) Nhắc lại công thức tính Accuracy cho bài toán phân loại đa lớp.

(c) Khi train tổng 5 epoch, accuracy có còn tăng không? Tăng nhanh hay chậm dần?

### 1.5. Bước 5 - Vẽ biểu đồ accuracy theo epoch

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_acc_history, 'o-', label='Train accuracy')
plt.plot(test_acc_history, 's-', label='Test accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('CNN học MNIST -- accuracy tăng theo epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Câu hỏi 5.1**

(a) Train accuracy và test accuracy cái nào cao hơn? Đây là điều bình thường hay bất thường?

(b) Ở epoch nào accuracy tăng nhanh nhất? Ở các epoch sau, đường cong có phẳng dần không?

### 1.6. Bước 6 - Xem từng dự đoán cụ thể

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = model(images)
    probs = F.softmax(outputs, dim=1)
    preds = outputs.argmax(1)

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    img = images[i, 0].cpu().numpy()
    pred, true = preds[i].item(), labels[i].item()
    conf = probs[i, pred].item() * 100
    color = 'green' if pred == true else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f"pred={pred} ({conf:.0f}%)\ntrue={true}", color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
wrong_imgs, wrong_pred, wrong_true = [], [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        mask = preds != labels
        for img, p, t in zip(images[mask], preds[mask], labels[mask]):
            wrong_imgs.append(img.cpu())
            wrong_pred.append(p.item())
            wrong_true.append(t.item())
        if len(wrong_imgs) >= 16:
            break

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    if i < len(wrong_imgs):
        ax.imshow(wrong_imgs[i][0], cmap='gray')
        ax.set_title(f"pred={wrong_pred[i]}\ntrue={wrong_true[i]}", color='red', fontsize=9)
    ax.axis('off')
plt.suptitle("Những ảnh CNN đoán SAI", color='red')
plt.tight_layout()
plt.show()

**Câu hỏi 6.1**

(a) Quan sát 16 ảnh đoán sai. Em có thể nhận ra số đó là gì không, kể cả khi là người? Có những ảnh viết rất xấu/rất kỳ không?

(b) Cặp số nào hay bị nhầm lẫn? Hai số đó có nét chữ giống nhau không?

(c) Mô hình sai theo cách có lý hay vô lý?

### 1.7. Bước 7 - Thử nghiệm: đổi số lớp của CNN

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv(x)))
        return self.fc(x.view(x.size(0), -1))


class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(64 * 3 * 3, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        return self.fc(x.view(x.size(0), -1))


def quick_train(model_class, epochs=3):
    torch.manual_seed(42)
    m = model_class().to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            crit(m(x), y).backward()
            opt.step()
    m.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item()
            total += y.size(0)
    n_params = sum(p.numel() for p in m.parameters())
    test_acc = 100 * correct / total
    print(f"{model_class.__name__:11s}: params = {n_params:>8,} | test_acc = {test_acc:.2f}%")
    return m, n_params, test_acc

print("Huấn luyện 3 kiến trúc khác nhau, 3 epoch mỗi mạng...\n")
shallow_model, shallow_params, shallow_acc = quick_train(ShallowCNN)
tiny_compare_model, tiny_params, tiny_acc = quick_train(TinyCNN)
deep_model, deep_params, deep_acc = quick_train(DeepCNN)

**Câu hỏi 7.1**

(a) Mạng nào nhiều tham số nhất? Là mạng nông, vừa, hay sâu?

(b) Mạng nào accuracy cao nhất? Có phải càng sâu càng tốt không? Hay có sự bão hòa?

(c) Tại sao mạng sâu hơn lại học tốt hơn? Thử đoán lý do.

### 1.8. Bước 8 - Thử nghiệm: bật/tắt augmentation

In [ ]:
transform_aug = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

sample_idx = 0
raw_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform_aug)
fig, axs = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axs):
    img, lbl = raw_set[sample_idx]
    ax.imshow(img[0], cmap='gray')
    ax.set_title(f"aug #{i+1}", fontsize=8)
    ax.axis('off')
plt.suptitle(f"1 ảnh số {lbl} -- 8 lần augment khác nhau")
plt.tight_layout()
plt.show()

In [ ]:
train_set_aug = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform_aug)
train_loader_aug = DataLoader(train_set_aug, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

m_aug = TinyCNN().to(device)
opt = optim.Adam(m_aug.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss()

for ep in range(3):
    m_aug.train()
    for x, y in train_loader_aug:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        crit(m_aug(x), y).backward()
        opt.step()

m_aug.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (m_aug(x).argmax(1) == y).sum().item()
        total += y.size(0)

mnist_aug_acc = 100 * correct / total
print(f"TinyCNN + augmentation: test_acc = {mnist_aug_acc:.2f}%")
print(f"TinyCNN không augment (so sánh): {test_acc_history[-1]:.2f}%")

**Câu hỏi 8.1**

(a) Khi xem 8 phiên bản augment của cùng một ảnh, các ảnh khác nhau hay giống nhau? Người vẫn dễ dàng nhận ra số bao nhiêu chứ?

(b) Trên MNIST, augmentation có giúp accuracy tăng nhiều không? Vì sao hiệu ứng có thể không lớn?

## 2. GIAI ĐOẠN 2 - Intuition phase: Tổng thể CNN

### 2.1. Trực giác 1: Kernel quét ảnh - Edge Detection thủ công

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
import urllib.request, io
import matplotlib.pyplot as plt
import numpy as np

url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    img = Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=20).read()))
except Exception as e:
    print("Không tải được ảnh từ Internet, dùng ảnh mẫu CIFAR-10 thay thế.")
    print(type(e).__name__, e)
    fallback_set = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)
    img = fallback_set[3][0]

img = img.convert('L').resize((256, 256))
img_t = torch.tensor(np.array(img), dtype=torch.float32)
img_t = img_t.unsqueeze(0).unsqueeze(0) / 255.0

kernels = {
    "Identity (không đổi)": torch.tensor([[0, 0, 0], [0, 1, 0], [0, 0, 0]]),
    "Cạnh ngang (Sobel-y)": torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Cạnh dọc (Sobel-x)": torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Làm mờ (box blur)": torch.tensor([[1, 1, 1], [1, 1, 1], [1, 1, 1]]) / 9.0,
    "Làm nét (sharpen)": torch.tensor([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Emboss": torch.tensor([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]]),
}

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
axs[0, 0].imshow(img, cmap='gray')
axs[0, 0].set_title("Ảnh gốc")
axs[0, 0].axis('off')

for ax, (name, K) in zip(axs.flat[1:], kernels.items()):
    K = K.float().view(1, 1, 3, 3)
    out_img = F.conv2d(img_t, K, padding=1).squeeze().numpy()
    ax.imshow(out_img, cmap='gray')
    ax.set_title(name)
    ax.axis('off')

for i in range(len(kernels) + 1, 8):
    axs.flat[i].axis('off')
plt.tight_layout()
plt.show()

**Câu hỏi 9.1**

(a) Nhìn vào kernel Sobel-y, đoán xem vì sao nó phát hiện được cạnh ngang?

(b) Tự thiết kế một kernel phát hiện đường chéo `/`.

(c) Trong CNN, các kernel này không phải do người tạo ra. Mạng tự học chúng từ dữ liệu. Cảm giác đó như thế nào?

### 2.2. Trực giác 2: Pooling - Nén ảnh nhưng giữ ý nghĩa

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(14, 6))
x = img_t.clone()
axs[0, 0].imshow(x[0, 0], cmap='gray')
axs[0, 0].set_title(f"Gốc -- {x.shape[-1]}x{x.shape[-1]}")
axs[1, 0].imshow(x[0, 0], cmap='gray')
axs[1, 0].set_title(f"Gốc -- {x.shape[-1]}x{x.shape[-1]}")

xm = img_t.clone()
for i in range(4):
    xm = F.max_pool2d(xm, 2)
    axs[0, i+1].imshow(xm[0, 0], cmap='gray')
    axs[0, i+1].set_title(f"MaxPool x{i+1}\n{xm.shape[-1]}x{xm.shape[-1]}")

xa = img_t.clone()
for i in range(4):
    xa = F.avg_pool2d(xa, 2)
    axs[1, i+1].imshow(xa[0, 0], cmap='gray')
    axs[1, i+1].set_title(f"AvgPool x{i+1}\n{xa.shape[-1]}x{xa.shape[-1]}")

for ax in axs.flat:
    ax.axis('off')
axs[0, 0].set_ylabel("MaxPool", fontsize=12)
axs[1, 0].set_ylabel("AvgPool", fontsize=12)
plt.suptitle("Pooling -- nén ảnh từng bước")
plt.tight_layout()
plt.show()

**Câu hỏi 10.1**

(a) Sau 4 lần pool, ảnh từ 256x256 co lại còn bao nhiêu? Em vẫn nhận ra đối tượng ở kích thước nhỏ nhất chứ?

(b) So sánh MaxPool và AvgPool: cái nào trông nét hơn, cái nào trông mờ hơn?

(c) Vì sao CNN dùng pooling?

### 2.3. Trực giác 3: Feature Hierarchy

In [ ]:
from torchvision import models
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

try:
    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).to(device).eval()
    print("Đã tải VGG-16 pretrained ImageNet.")
except Exception as e:
    print("Không tải được VGG-16 pretrained, dùng weights=None để notebook vẫn chạy.")
    print(type(e).__name__, e)
    vgg = models.vgg16(weights=None).to(device).eval()

print(vgg.features)

transform_imnet = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    img_rgb = Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=20).read())).convert('RGB')
except Exception:
    fallback_set = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)
    img_rgb = fallback_set[3][0].convert('RGB')

x = transform_imnet(img_rgb).unsqueeze(0).to(device)
print("Input shape:", x.shape)

In [ ]:
features = {}

def make_hook(name):
    def hook(module, inp, out):
        features[name] = out.detach()
    return hook

h1 = vgg.features[0].register_forward_hook(make_hook("block1_conv1"))
h2 = vgg.features[10].register_forward_hook(make_hook("block3_conv1"))
h3 = vgg.features[24].register_forward_hook(make_hook("block5_conv1"))

with torch.no_grad():
    _ = vgg(x)

h1.remove()
h2.remove()
h3.remove()

for k, v in features.items():
    print(f"{k}: feature map shape = {tuple(v.shape)}")

In [ ]:
def show_feature_maps(fmap, title, n_show=8):
    fmap = fmap[0].cpu()
    channel_means = fmap.abs().mean(dim=(1, 2))
    top_idx = channel_means.argsort(descending=True)[:n_show]
    fig, axs = plt.subplots(1, n_show, figsize=(16, 2.4))
    for i, ax in enumerate(axs):
        ax.imshow(fmap[top_idx[i]], cmap='viridis')
        ax.set_title(f"ch {top_idx[i].item()}", fontsize=9)
        ax.axis('off')
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

show_feature_maps(features["block1_conv1"], "block1_conv1 (LỚP NÔNG) -- cạnh và vùng màu")
show_feature_maps(features["block3_conv1"], "block3_conv1 (LỚP GIỮA) -- texture và pattern")
show_feature_maps(features["block5_conv1"], "block5_conv1 (LỚP SÂU) -- bộ phận, khái niệm trừu tượng")

**Câu hỏi 11.1**

(a) In ra shape của feature map ở 3 lớp. Càng đi sâu, kích thước không gian H, W thay đổi thế nào? Số kênh C thay đổi thế nào?

(b) Ở `block1_conv1`, trong 8 channel hiển thị có bao nhiêu channel giống phát hiện cạnh ngang, cạnh dọc hoặc vùng màu sáng?

(c) Nếu thay ảnh đầu vào, feature map thay đổi như thế nào ở lớp nông và lớp sâu?

### 2.4. Bonus: CNN Explainer

Truy cập: https://poloclub.github.io/cnn-explainer/

**Câu hỏi 12.1**

(a) Mô tả bằng lời cách một phép tích chập diễn ra.

(b) Khi đi từ Conv1 sang Conv2, vì sao feature map thu nhỏ dần?

(c) Lớp softmax cuối cùng làm gì?

(d) Có khái niệm nào trong CNN Explainer mà em chưa hiểu rõ?

## 3. Bài tập về nhà đã chọn: Bài 1 - Lặp lại WOW phase trên CIFAR-10

In [ ]:
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_train_set = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform)
cifar_test_set = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=cifar_transform)

cifar_train_loader = DataLoader(cifar_train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
cifar_test_loader = DataLoader(cifar_test_set, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(cifar_train_loader))
print(f"CIFAR batch images shape: {images.shape}")
print(f"CIFAR batch labels shape: {labels.shape}")
print("Train size:", len(cifar_train_set))
print("Test size:", len(cifar_test_set))

cifar_mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
cifar_std = torch.tensor((0.2470, 0.2435, 0.2616)).view(3, 1, 1)

def denorm_cifar(x):
    return (x.cpu() * cifar_std + cifar_mean).clamp(0, 1)

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    ax.imshow(denorm_cifar(images[i]).permute(1, 2, 0).numpy())
    ax.set_title(cifar_classes[labels[i].item()], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class CifarTinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

cifar_model = CifarTinyCNN().to(device)
print(cifar_model)
print(f"Tổng tham số: {sum(p.numel() for p in cifar_model.parameters()):,}")

In [ ]:
def train_cifar_model(model, train_loader, test_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_acc_history, test_acc_history = [], []

    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_acc = 100 * correct / total

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += labels.size(0)
        test_acc = 100 * correct / total
        train_acc_history.append(train_acc)
        test_acc_history.append(test_acc)
        print(f"Epoch {epoch+1}: train_acc = {train_acc:.2f}% | test_acc = {test_acc:.2f}%")

    return train_acc_history, test_acc_history

cifar_train_acc_history, cifar_test_acc_history = train_cifar_model(cifar_model, cifar_train_loader, cifar_test_loader, epochs=5, lr=0.001)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(cifar_train_acc_history, 'o-', label='Train accuracy')
plt.plot(cifar_test_acc_history, 's-', label='Test accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('CNN học CIFAR-10 -- accuracy theo epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cifar_model.eval()
images, labels = next(iter(cifar_test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = cifar_model(images)
    probs = F.softmax(outputs, dim=1)
    preds = outputs.argmax(1)

fig, axs = plt.subplots(2, 8, figsize=(15, 4.5))
for i, ax in enumerate(axs.flat):
    img = denorm_cifar(images[i].cpu()).permute(1, 2, 0).numpy()
    pred, true = preds[i].item(), labels[i].item()
    conf = probs[i, pred].item() * 100
    color = 'green' if pred == true else 'red'
    ax.imshow(img)
    ax.set_title(f"pred={cifar_classes[pred]} ({conf:.0f}%)\ntrue={cifar_classes[true]}", color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
wrong_imgs, wrong_pred, wrong_true = [], [], []
cifar_model.eval()
with torch.no_grad():
    for images, labels in cifar_test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = cifar_model(images).argmax(1)
        mask = preds != labels
        for img, p, t in zip(images[mask], preds[mask], labels[mask]):
            wrong_imgs.append(img.cpu())
            wrong_pred.append(p.item())
            wrong_true.append(t.item())
        if len(wrong_imgs) >= 16:
            break

fig, axs = plt.subplots(2, 8, figsize=(15, 4.5))
for i, ax in enumerate(axs.flat):
    if i < len(wrong_imgs):
        img = denorm_cifar(wrong_imgs[i]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(f"pred={cifar_classes[wrong_pred[i]]}\ntrue={cifar_classes[wrong_true[i]]}", color='red', fontsize=8)
    ax.axis('off')
plt.suptitle("Những ảnh CIFAR-10 đoán SAI", color='red')
plt.tight_layout()
plt.show()

In [ ]:
cifar_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_train_set_aug = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform_aug)
cifar_train_loader_aug = DataLoader(cifar_train_set_aug, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

preview_aug = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform_aug)
fig, axs = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axs):
    img, lbl = preview_aug[0]
    ax.imshow(denorm_cifar(img).permute(1, 2, 0).numpy())
    ax.set_title(f"aug #{i+1}", fontsize=8)
    ax.axis('off')
plt.suptitle(f"1 ảnh CIFAR-10 lớp {cifar_classes[lbl]} -- 8 lần augment khác nhau")
plt.tight_layout()
plt.show()

cifar_model_aug = CifarTinyCNN().to(device)
cifar_aug_train_acc_history, cifar_aug_test_acc_history = train_cifar_model(cifar_model_aug, cifar_train_loader_aug, cifar_test_loader, epochs=5, lr=0.001)

print(f"CIFAR-10 không augment: {cifar_test_acc_history[-1]:.2f}%")
print(f"CIFAR-10 có augment: {cifar_aug_test_acc_history[-1]:.2f}%")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(cifar_train_acc_history, 'o-', label='Train no aug')
plt.plot(cifar_test_acc_history, 's-', label='Test no aug')
plt.plot(cifar_aug_train_acc_history, 'o--', label='Train aug')
plt.plot(cifar_aug_test_acc_history, 's--', label='Test aug')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('CIFAR-10: so sánh có/không augmentation')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
summary_df = pd.DataFrame([
    {'Thí nghiệm': 'MNIST TinyCNN', 'Epochs': 5, 'Test accuracy (%)': test_acc_history[-1]},
    {'Thí nghiệm': 'MNIST TinyCNN + augmentation', 'Epochs': 3, 'Test accuracy (%)': mnist_aug_acc},
    {'Thí nghiệm': 'MNIST ShallowCNN', 'Epochs': 3, 'Params': shallow_params, 'Test accuracy (%)': shallow_acc},
    {'Thí nghiệm': 'MNIST TinyCNN so sánh', 'Epochs': 3, 'Params': tiny_params, 'Test accuracy (%)': tiny_acc},
    {'Thí nghiệm': 'MNIST DeepCNN', 'Epochs': 3, 'Params': deep_params, 'Test accuracy (%)': deep_acc},
    {'Thí nghiệm': 'CIFAR-10 CNN', 'Epochs': 5, 'Test accuracy (%)': cifar_test_acc_history[-1]},
    {'Thí nghiệm': 'CIFAR-10 CNN + augmentation', 'Epochs': 5, 'Test accuracy (%)': cifar_aug_test_acc_history[-1]},
])
display(summary_df)

**Câu hỏi cho Bài 1 CIFAR-10**

1. CIFAR-10 có shape ảnh khác MNIST như thế nào?

2. Accuracy sau 5 epoch là bao nhiêu? Có thấp hơn MNIST không?

3. Trong 16 ảnh đoán sai, ảnh nào khó nhận ra ngay cả với người?

4. Augmentation `RandomHorizontalFlip` và `RandomCrop` có cải thiện accuracy không?